# Étude de cas : les tuyaux de Nikuradse, en entier

En 1933, Johann Nikuradse a collé des grains de sable de taille connue à
l'intérieur de tuyaux et mesuré le frottement de l'écoulement : 362 mesures, six
rapports de rugosité, des nombres de Reynolds d'environ 4 000 à 1 000 000. On ne
connaît pas de formule exacte pour toute cette plage. La relation de Prandtl et
von Kármán des manuels, sans aucun paramètre ajusté, couvre le régime
pleinement rugueux et explique 94 % de la variance de ces mesures.

Ce notebook déroule le protocole complet de `benchmarks/real_nikuradse.py` : un
front de formules jugées sur des lignes tenues à l'écart de l'ajustement, puis
un rapport de rugosité entier retiré de l'ajustement et prédit. GP_ELITE
propose des formes courtes qui ajustent presque aussi bien que la relation des
manuels, et des formes longues qui ajustent mieux ; aucune des formes courtes ne
prédit la rugosité retirée aussi bien que la relation des manuels.

Chaque chiffre du texte a été mesuré en lançant ce script pour les graines 0 à 4
(sortie dans `benchmarks/results_0.9/case_studies/nikuradse_seed*.txt`).

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Les deux recherches
prennent environ une minute chacune.

*[🇬🇧 English version](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/nikuradse_full.ipynb)*

## 1. Installation

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'prêt.')

## 2. Les données et leur source

Les mesures viennent de PMLB, la collection Penn Machine Learning Benchmarks
(jeu `nikuradse_1`), comme dans le notebook de prise en main : le rapport de
rugosité `r_k` (rayon du tuyau sur taille du grain), le logarithme décimal du
nombre de Reynolds `log_Re`, et le frottement sous la forme
`log10(100 * lambda)`. Les bancs de ce projet vérifient le fichier par son
empreinte SHA-256 (`benchmarks/pmlb_frozen.py`).

In [ ]:
import numpy as np
import pandas as pd

URL = ('https://github.com/EpistasisLab/pmlb/raw/master/datasets/'
       'nikuradse_1/nikuradse_1.tsv.gz')
df = pd.read_csv(URL, sep='\t', compression='gzip')

X = df[['r_k', 'log_Re']].to_numpy(float)
y = df['target'].to_numpy(float)
print(len(y), 'mesures, rapports de rugosité', sorted(df['r_k'].unique().tolist()))

## 3. Les références

Les lignes sont partagées au hasard, 70 % pour ajuster et 30 % pour tester,
toujours de la même façon. Trois références sont notées sur les lignes de test :
une droite en `r_k` et `log_Re` ; une droite en `log10(r_k)` et `log_Re`, trois
paramètres ajustés ; et la relation de Prandtl et von Kármán, qui s'écrit pour
cette cible `2 - 2*log10(2*log10(r_k) + 1.74)`, sans aucun paramètre ajusté.

In [ ]:
rng = np.random.RandomState(0)
idx = rng.permutation(len(y))
ntr = int(0.7 * len(y))
tr, te = idx[:ntr], idx[ntr:]

def r2(p, t):
    return 1 - np.mean((p - t) ** 2) / np.var(t)

def pvk(Z):     # Prandtl-von Karman, régime pleinement rugueux
    return 2.0 - 2.0 * np.log10(2.0 * np.log10(np.abs(Z[:, 0]) + 1e-12) + 1.74)

def line(basis, itr, ite):     # moindres carrés sur une base, R2 sur ite
    A = np.c_[np.ones(len(itr)), basis(X[itr])]
    coef = np.linalg.lstsq(A, y[itr], rcond=None)[0]
    return r2(np.c_[np.ones(len(ite)), basis(X[ite])] @ coef, y[ite])

logs = lambda Z: np.c_[np.log10(np.abs(Z[:, 0]) + 1e-12), Z[:, 1]]
print(f'{len(tr)} lignes pour ajuster, {len(te)} pour tester')
print(f'droite, ajustée                 R2 test {line(lambda Z: Z, tr, te):.4f}')
print(f'droite en log10(r_k), ajustée   R2 test {line(logs, tr, te):.4f}')
print(f'Prandtl-von Karman, sans ajuster R2 test {r2(pvk(X[te]), y[te]):.4f}'
      f'   (toutes les lignes {r2(pvk(X), y):.4f})')

Sur les 109 lignes de test : la droite R² 0,6778, la droite en `log10(r_k)`
0,9346, la relation de Prandtl et von Kármán 0,9456 (0,9434 sur les 362 lignes).
C'est cette dernière la barre.

## 4. La recherche, et son front

`generations=30`, quatre relances, 15 % des lignes d'ajustement gardées pour la
validation interne du moteur. Le tableau donne le front de Pareto, la meilleure
formule pour chaque taille, notée sur les lignes de test.

In [ ]:
import time
from gp_elite import symbolic_regression

def search(Xf, yf):
    return symbolic_regression(Xf, yf, feature_names=['r_k', 'log_Re'],
                               operators='physical', normalize='auto',
                               generations=30, speed='fast',
                               validation_split=0.15, seed=0, restarts=4,
                               parallel=False)

def front(model, Zt, yt):
    rows = {}
    for e in list(model.pareto or []) + [model]:
        rows.setdefault((int(e.size), round(r2(e.predict(Zt), yt), 6)), e)
    return sorted(rows.items(), key=lambda kv: kv[0])

t0 = time.time()
m = search(X[tr], y[tr])
print(f'recherche : {time.time() - t0:.0f} s\n')
print('taille  R2 test  expression')
for (s, r), e in front(m, X[te], y[te]):
    print(f'{s:>6}  {r:7.4f}  {e.expression[:90]}')

**Ce que rendent les cinq graines.** Les cinq commencent le front par la même
forme à six nœuds, `1.10108 - 0.133625 * log(r_k)`, R² 0,9357 sur les lignes de
test. Les graines 0, 2 et 4 rendent aussi une puissance de la rugosité, de sept
à neuf nœuds : 1,559·r_k^(−1/4) − 0,0282 (graines 0 et 4, écrite
`sqrt(1 / sqrt(r_k))` ou `sqrt(sqrt(r_k) / r_k)`, R² 0,9447), et
0,0035 + 1,565·r_k^(−0,266) (graine 2, R² 0,9441) : à moins de 0,0015 de la
relation des manuels sur les lignes de test. Les graines 1 et 3 rendent des
formes en `log(tanh(...))` à sept et huit nœuds, R² de 0,9404 à 0,9426.

Le champion rendu, de 48 à 70 nœuds, atteint un R² de 0,980 à 0,995 sur les
lignes de test. Il suit la dépendance au nombre de Reynolds, que la relation des
manuels laisse de côté, sous une forme que personne ne lirait comme une loi. Sur
des données réelles, la réponse lisible est dans le front, pas dans le
champion. Le moteur peut aussi avertir qu'une formule imprimée s'écarte de ses
prédictions sur quelques lignes, quand un filet de sécurité du moteur y agit ;
les R² du tableau sont ceux des prédictions.

## 5. Le vrai test : une rugosité jamais vue

`r_k` ne prend que six valeurs : les données sont six courbes. Un partage au
hasard teste des points à l'intérieur de courbes que le moteur a déjà vues. Ici
la rugosité 126 est entièrement retirée de l'ajustement, et chaque forme du
nouveau front la prédit.

Une loi de `r_k` seul prédit une ligne plate le long de la courbe retirée, dont
les mesures varient avec le nombre de Reynolds ; le R² y dit peu de chose. La
mesure est le niveau : l'écart entre la moyenne prédite et la moyenne mesurée
sur la courbe retirée, en écarts-types de cette courbe.

In [ ]:
out = np.isclose(X[:, 0], 126.0)
Xin, yin, Xout, yout = X[~out], y[~out], X[out], y[out]

def bias(pred):
    return (np.mean(pred) - np.mean(yout)) / np.std(yout)

t0 = time.time()
m2 = search(Xin, yin)
print(f'recherche : {time.time() - t0:.0f} s')
print(f'{len(yin)} lignes pour ajuster (cinq rugosités), {len(yout)} à prédire\n')
print(f'Prandtl-von Karman : écart de niveau {bias(pvk(Xout)):+.3f} écart-type')
print('taille  écart de niveau  expression')
for (s, r), e in front(m2, Xout, yout):
    print(f'{s:>6}  {bias(e.predict(Xout)):+.3f}            {e.expression[:70]}')

La relation des manuels manque le niveau de la courbe retirée de 0,042
écart-type, sans aucun choix à faire. La forme à six nœuds le manque de 0,225
dans les cinq graines. Sur les cinq graines, le moteur propose de 10 à 13
formes, dont les écarts vont de −1,20 à +0,88. La meilleure de chaque graine est
à 0,014 à 0,225 du niveau, et seules deux formes font mieux que la relation des
manuels, toutes deux de la graine 0, longues de 52 et 59 nœuds ; mais la
meilleure ne peut être choisie qu'après avoir vu la courbe retirée. Avant, rien
ne dit laquelle prendre.

## 6. Ce que le moteur ne trouve pas

- **La relation des manuels elle-même.** Un logarithme d'un logarithme de
  `r_k`, la famille de Prandtl et von Kármán, apparaît dans un front sur cinq, à
  l'intérieur d'une forme de 30 nœuds (graine 1). Les formes courtes que le
  moteur propose ajustent presque aussi bien sur les lignes de test, et
  prédisent moins bien la rugosité retirée.
- **Une loi lisible du régime de transition.** Les formes qui suivent le nombre
  de Reynolds sont longues ; c'est la partie du problème qui reste ouverte.
- **Ce qui se généralise.** Seul un test tenu à l'écart, comme à la section 5,
  dit jusqu'où une forme porte. Le moteur ne peut pas le dire à partir des seules
  lignes ajustées.

## 7. Les cinq graines

Les sections 4 et 5 donnent l'étendue sur les graines 0 à 4 ; le notebook lance
la graine 0. Les chiffres ont été mesurés sur la
branche de développement qui suit la version 0.9.0, puis de nouveau sur le
paquet de la version 1.0.0, avec les mêmes résultats au dernier chiffre affiché
près (`benchmarks/results_0.9/release_1.0/m3/`).

La sortie complète, un fichier par graine : `benchmarks/results_0.9/case_studies/`.
Le protocole, écrit avant les mesures : `benchmarks/results_0.9/PLAN_PHASE6.md`.